# 🧗 Day 19: Cliff Walking - Q-Learning vs. SARSA Head-to-Head
### 100 Days of Machine Learning | Syed Muhammad Qamran
**Topic:** On-Policy vs. Off-Policy Control in High-Risk Environments  
**Benchmark:** Sutton & Barto Example 6.6 | Gymnasium `CliffWalking-v1`

---

### The Fundamental Dilemma:
- **Q-Learning (Off-policy):** Discovers the mathematically optimal shortest path directly adjacent to the cliff (-13 reward). However, during exploration ($\epsilon$-greedy), it occasionally falls off the cliff (-100 penalty), yielding poorer online training performance.
- **SARSA (On-policy):** Realizes that with $\epsilon > 0$, taking the risky edge leads to fatal falls. It learns the **safe, conservative path** around the top row (-17 reward), yielding much higher cumulative reward during training!


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline


In [ ]:
# Load Cliff Walking environment
env = gym.make("CliffWalking-v1")

class CliffAgent:
    def __init__(self, mode="q_learning", alpha=0.5, gamma=1.0, epsilon=0.1):
        self.mode = mode
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((48, 4), dtype=np.float64)

    def get_action(self, s, greedy=False):
        if not greedy and np.random.uniform(0, 1) < self.epsilon:
            return np.random.randint(4)
        max_q = np.max(self.Q[s, :])
        return int(np.random.choice(np.where(self.Q[s, :] == max_q)[0]))

    def update(self, s, a, r, ns, na=None):
        if self.mode == "q_learning":
            target = r + self.gamma * np.max(self.Q[ns, :])
        else: # sarsa
            target = r + self.gamma * self.Q[ns, na]
        self.Q[s, a] += self.alpha * (target - self.Q[s, a])

def run_experiment(mode, episodes=500):
    agent = CliffAgent(mode=mode)
    ep_rewards = []
    for ep in range(episodes):
        s, _ = env.reset()
        a = agent.get_action(s)
        total_r = 0
        for _ in range(500):
            ns, r, done, trunc, _ = env.step(a)
            na = agent.get_action(ns)
            agent.update(s, a, r, ns, na)
            s, a = ns, na
            total_r += r
            if done or trunc:
                break
        ep_rewards.append(total_r)
    return agent, ep_rewards

q_agent, q_rewards = run_experiment("q_learning")
sarsa_agent, sarsa_rewards = run_experiment("sarsa")

# Plot smoothed learning curves
def smooth(arr, w=15):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(smooth(sarsa_rewards), label="SARSA (Safe Path)", color="#10B981", lw=2)
ax.plot(smooth(q_rewards), label="Q-Learning (Optimal Path but Falls)", color="#EF4444", lw=2)
ax.set_title("Cliff Walking: Q-Learning vs. SARSA Reward Comparison", fontsize=13, fontweight='bold')
ax.set_xlabel("Episode")
ax.set_ylabel("Sum of Rewards during Episode")
ax.set_ylim(-150, 0)
ax.legend(fontsize=11)
plt.show()
